<a href="https://colab.research.google.com/github/SsemuliJoseph/Sunbird/blob/main/Data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install argilla datasets huggingface_hub

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 2.5 MB/s eta 0:00:00
  Created wheel for standardwebhooks: filename=standardwebhooks-1.0.1-py3-none-any.whl size=3619 sha256=4d3a1ed6d0c5583cec5d8887d9a0eb8937b71e2895f62f91edccd713cf58ccbc
  Stored in directory: /root/.cache/pip/wheels/99/f1/9e/5875dab65df60a0e530f19d477880f0276a343e1594fb3ee5f
Successfully built standardwebhooks


In [ ]:
import argilla as rg
from datasets import load_dataset
from google.colab import userdata
import huggingface_hub

# 1. Securely fetch tokens from Colab Secrets
my_argilla_key = userdata.get('ARGILLA_API_KEY')
my_hf_token = userdata.get('HF_TOKEN')

# 2. Authenticate with the Hugging Face Hub
print("Authenticating with Hugging Face Hub...")
huggingface_hub.login(token=my_hf_token)

# 3. Connect to Argilla securely
client = rg.Argilla(
    api_url="https://ssemulijoseph-nlp-data-curation.hf.space",
    api_key=my_argilla_key
)

# 4. Load the REAL dataset from Hugging Face (Now authenticated!)
print("Downloading real dataset from the Hub...")
real_dataset = load_dataset("kambale/luganda-english-parallel-corpus", split="train")

# Take a sample of 100 rows to avoid overloading the free tier server
sampled_dataset = real_dataset.select(range(100))

# 5. Define the workspace structure in Argilla
settings = rg.Settings(
    fields=[
        rg.TextField(name="english", title="English Source"),
        rg.TextField(name="luganda", title="Luganda Translation")
    ],
    questions=[
        rg.LabelQuestion(
            name="quality",
            title="Is this translation accurate?",
            labels=["Yes", "No", "Needs Editing"]
        ),
        rg.TextQuestion(
            name="correction",
            title="If incorrect, provide the right translation:",
            required=False
        )
    ]
)

# 6. Create the dataset on the server and push the records
dataset = rg.Dataset(
    name="real_luganda_curation_v1",
    settings=settings,
    client=client,
)
dataset.create()

# Log the data into the Argilla UI
dataset.records.log(sampled_dataset)
print("Success! Authenticated pipeline complete and data pushed.")

Authenticating with Hugging Face Hub...


luganda-english-parallel-corpus.json:   0%|          | 0.00/4.22M [00:00<?, ?B/s]

luganda-english-parallel-corpus.jsonl:   0%|          | 0.00/3.55M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/50012 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/argilla/datasets/_resource.py:264: UserWarning: Workspace not provided. Using default workspace: argilla id: 55679cb5-eb9f-4d31-8301-3706449c2f13
  warnings.warn(f"Workspace not provided. Using default workspace: {workspace.name} id: {workspace.id}")
/usr/local/lib/python3.12/dist-packages/argilla/records/_io/_datasets.py:265: UserWarning: Record id column not found in Hugging Face dataset. Using row index and split for record ids.
  warnings.warn(


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

DatasetRecords: The provided batch size 256 was normalized. Using value 100.

Sending records...: 100%|██████████| 1/1 [00:02<00:00,  2.02s/batch]

Success! Authenticated pipeline complete and data pushed.


In [ ]:
!pip install transformers[torch] datasets evaluate sacrebleu accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 9.1 MB/s eta 0:00:00


In [ ]:
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)
import evaluate
import numpy as np

# 1. Hardware Check (Ensure this prints "Using device: cuda" now!)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# 2. Configurations & Model Selection
MODEL_CHECKPOINT = "facebook/nllb-200-distilled-600M"
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "lug_Latn"
MAX_LENGTH = 128

# 3. Load and Split the Real Dataset
print("Loading dataset splits...")
raw_dataset = load_dataset("kambale/luganda-english-parallel-corpus", split="train")

dataset_split = raw_dataset.train_test_split(test_size=200, train_size=2000, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# 4. Initialize the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, src_lang=SOURCE_LANG, tgt_lang=TARGET_LANG)

# 5. Fixed Preprocessing & Tokenization Function
def preprocess_function(examples):
    inputs = examples["english"]
    targets = examples["luganda"]

    # Industrial Approach: Use text_target directly instead of the old context manager
    model_inputs = tokenizer(
        inputs,
        text_target=targets,
        max_length=MAX_LENGTH,
        truncation=True
    )

    return model_inputs

print("Tokenizing datasets...")
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_eval = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

# 6. Load the Pre-trained Model
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_CHECKPOINT).to(device)

# 7. Define Evaluation Metric (BLEU Score)
metric = evaluate.load("sacrebleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    # Replace -100 placeholders so we can safely decode the text labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)

    result = metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {"bleu": result["score"]}

# 8. Data Collator
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

# 9. Set Industrial Training Arguments
# 9. Set Industrial Training Arguments (Fixed for latest Transformers API)
training_args = Seq2SeqTrainingArguments(
    output_dir="./nllb-luganda-baseline",
    eval_strategy="epoch",    # <-- CHANGED FROM evaluation_strategy TO eval_strategy
    learning_rate=5e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=1,
    num_train_epochs=1,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

# 10. Initialize the Trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 11. Start Training
print("Starting training loop...")
trainer.train()

print("Evaluation results:")
print(trainer.evaluate())

Using device: cuda
Loading dataset splits...
Tokenizing datasets...


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Starting training loop...


Epoch,Training Loss,Validation Loss,Bleu
1,1.625723,1.528900,14.643844


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluation results:


Training Loss,Validation Loss,Epoch,Bleu
1.625723,1.528900,1,14.643844


{'eval_loss': 1.528900146484375, 'eval_bleu': 14.643843889642923}


In [ ]:
!pip install argilla

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.3/161.3 kB 6.5 MB/s eta 0:00:00
  Created wheel for standardwebhooks: filename=standardwebhooks-1.0.1-py3-none-any.whl size=3619 sha256=aa78f99fd620a61793082dac02f0481be77f9ffa6c667ce23588be35ac5df3b5
  Stored in directory: /root/.cache/pip/wheels/99/f1/9e/5875dab65df60a0e530f19d477880f0276a343e1594fb3ee5f
Successfully built standardwebhooks


In [ ]:
import argilla as rg
from google.colab import userdata
# from datasets import load_dataset # Removed as it's not used for this loading method

# 1. Connect to your Space securely
my_api_key = userdata.get('ARGILLA_API_KEY')
client = rg.Argilla(
    api_url="https://ssemulijoseph-nlp-data-curation.hf.space",
    api_key=my_api_key
)

# 2. Load the dataset from Argilla directly into a Hugging Face Dataset format
print("Loading dataset from Argilla into Hugging Face Dataset format...")
# Use client.datasets() to retrieve the Argilla dataset object (v2 style),
# then convert its records to a Hugging Face Dataset using .records.to_datasets()
argilla_dataset_v2_object = client.datasets(name="real_luganda_curation_v1")
hf_backed_up_dataset = argilla_dataset_v2_object.records.to_datasets()

# 3. The dataset is now a Hugging Face Dataset, which can be iterated directly
records = hf_backed_up_dataset

# 4. Filter and check how many you have actually answered
# For a Hugging Face Dataset loaded from Argilla, 'quality.responses' is a column.
# We check if this column exists and has non-empty values (i.e., has been annotated).
# Using str() and checking length to account for potential string representations of lists.
completed_annotations = [r for r in records if r.get('quality.responses') is not None and len(str(r.get('quality.responses'))) > 2]
print(f"Total Dataset Rows: {len(records)}")
print(f"Human Curated Rows: {len(completed_annotations)}")

# 5. The hf_backed_up_dataset is already in Hugging Face Dataset format.

# 6. Save a local backup file inside Colab just in case
hf_backed_up_dataset.to_csv("argilla_luganda_backup.csv")
print("Local CSV backup saved inside Colab files!")

# Optional: Push it to your private HF profile as an ultimate safeguard
# hf_backed_up_dataset.push_to_hub("your-username/luganda-clean-gold-standard", private=True)

Loading dataset from Argilla into Hugging Face Dataset format...
Total Dataset Rows: 100
Human Curated Rows: 7


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Local CSV backup saved inside Colab files!


In [ ]:
import argilla as rg
from google.colab import userdata

# 1. Connect to your Space securely
my_api_key = userdata.get('ARGILLA_API_KEY')
client = rg.Argilla(
    api_url="https://ssemulijoseph-nlp-data-curation.hf.space",
    api_key=my_api_key
)

# 2. Retrieve the Argilla v2 dataset object
print("Connecting to your Argilla dataset...")
dataset = client.datasets(name="real_luganda_curation_v1")

# 3. Export the records directly into a Hugging Face Dataset format
print("Pulling records and converting to Hugging Face Dataset format...")
hf_backed_up_dataset = dataset.records.to_datasets()

# 4. Check your curation progress
total_rows = len(hf_backed_up_dataset)
print(f"Total Dataset Rows pulled: {total_rows}")

# 5. Save a local backup file inside Colab just in case
hf_backed_up_dataset.to_csv("argilla_luganda_backup.csv")
print("🎉 Success! Local CSV backup saved inside Colab files.")

# Optional: Push it to your private HF profile as an ultimate safeguard
# hf_backed_up_dataset.push_to_hub("your-username/luganda-clean-gold-standard", private=True)

Connecting to your Argilla dataset...
Pulling records and converting to Hugging Face Dataset format...
Total Dataset Rows pulled: 100


Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

🎉 Success! Local CSV backup saved inside Colab files.


In [ ]:
import pandas as pd
import json

# Load the backup CSV
df = pd.read_csv("argilla_luganda_backup.csv")

# Filter for rows where a human response exists
# In Argilla v2 exports, responses are stored as JSON strings or lists
df_clean = df[df['quality.responses'].notna()]

print(f"Verified rows with human annotations: {len(df_clean)}")
if len(df_clean) > 0:
    # Display the English source, Luganda translation, and your quality rating
    print(df_clean[['english', 'luganda', 'quality.responses']].head())

Verified rows with human annotations: 7
                                              english  \
5   Where can we get good resistant varieties that...   
19  Every Monday  farmers are given advice on coff...   
25  Farmers are given a platform to ask any agricu...   
35  Farming programs aired on radio help farmers t...   
41   Maize leaf rust causes the tip of a leaf to dry.   

                                              luganda quality.responses  
5   Tusobola kuggya wa ebika ebirungi ebigumira em...           ['Yes']  
19   Buli Mande abalimi bawabulwa ku nnima y'emmwanyi           ['Yes']  
25  Abalimi baweebwa omwagaanya okubuuza ekibuuzo ...           ['Yes']  
35  Pulogulaamu z'ebyobulimi eziweerezebwa ku laad...           ['Yes']  
41  Okumyukirira kw'ebikoola bya kasooli kireetera...           ['Yes']  


In [ ]:
import pandas as pd
import ast

def extract_golden_dataset(csv_path):
    df = pd.read_csv(csv_path)

    clean_pairs = []

    for idx, row in df.iterrows():
        # Check if the row has been reviewed by a human
        if pd.isna(row['quality.responses']):
            continue

        try:
            # Safely convert the string representation of a list back to a real list
            quality_list = ast.literal_eval(row['quality.responses'])
            quality = quality_list[0] if quality_list else "No"
        except (ValueError, SyntaxError):
            continue

        english_text = row['english']

        # Strategy 1: If it's perfect, take the original Luganda translation
        if quality == "Yes":
            luganda_text = row['luganda']
            clean_pairs.append({"english": english_text, "luganda": luganda_text, "source": "human_verified"})

        # Strategy 2: If it needed edits, extract your manual human correction
        elif quality == "Needs Editing":
            if pd.notna(row['correction.responses']):
                try:
                    correction_list = ast.literal_eval(row['correction.responses'])
                    luganda_text = correction_list[0] if correction_list else row['luganda']
                    clean_pairs.append({"english": english_text, "luganda": luganda_text, "source": "human_corrected"})
                except (ValueError, SyntaxError):
                    continue

        # Strategy 3: If you marked it "No" (completely unsalvageable), we drop it entirely
        elif quality == "No":
            continue

    return pd.DataFrame(clean_pairs)

# Execute the extraction
golden_df = extract_golden_dataset("argilla_luganda_backup.csv")

print(f"📊 Golden Dataset Size: {len(golden_df)} rows")
print("---")
if not golden_df.empty:
    print(golden_df[['english', 'luganda', 'source']].head())

📊 Golden Dataset Size: 7 rows
---
                                             english  \
0  Where can we get good resistant varieties that...   
1  Every Monday  farmers are given advice on coff...   
2  Farmers are given a platform to ask any agricu...   
3  Farming programs aired on radio help farmers t...   
4   Maize leaf rust causes the tip of a leaf to dry.   

                                             luganda          source  
0  Tusobola kuggya wa ebika ebirungi ebigumira em...  human_verified  
1   Buli Mande abalimi bawabulwa ku nnima y'emmwanyi  human_verified  
2  Abalimi baweebwa omwagaanya okubuuza ekibuuzo ...  human_verified  
3  Pulogulaamu z'ebyobulimi eziweerezebwa ku laad...  human_verified  
4  Okumyukirira kw'ebikoola bya kasooli kireetera...  human_verified  
